In [0]:
df = spark.read.json("/Volumes/my_files/my_schema/sample_datasets/json_files_uploaded/sales_corrupted.json")

df.show()

+--------------------+-------------+----+-----+-----------------+--------+--------------------+
|     _corrupt_record|customer_name|  id|price|          product|quantity|             remarks|
+--------------------+-------------+----+-----+-----------------+--------+--------------------+
|                NULL|     John Doe|   1|85000| Laptop, 16GB RAM|       2|   Delivered on time|
|                NULL|   Jane Smith|   2|55000|Smartphone, 128GB|       1|Customer said "ex...|
|{"id":3,"customer...|         NULL|NULL| NULL|             NULL|    NULL|                NULL|
|                NULL|   Alice Wong|   4| 2500|         Keyboard|    NULL|    Missing quantity|
|{"id":5,"customer...|         NULL|NULL| NULL|             NULL|    NULL|                NULL|
|                NULL|    Tom Brown|   6|  700|            Mouse|       5|Delivered, no issues|
|                NULL|   Mark Davis|   7|  150|        USB Cable|       2|                    |
+--------------------+-------------+----

In [0]:
df = spark.read.format("json")\
.option("mode", "DROPMALFORMED")\
.load("/Volumes/my_files/my_schema/sample_datasets/json_files_uploaded/sales_corrupted.json")

df.show()

+-------------+---+-----+-----------------+--------+--------------------+
|customer_name| id|price|          product|quantity|             remarks|
+-------------+---+-----+-----------------+--------+--------------------+
|     John Doe|  1|85000| Laptop, 16GB RAM|       2|   Delivered on time|
|   Jane Smith|  2|55000|Smartphone, 128GB|       1|Customer said "ex...|
|   Alice Wong|  4| 2500|         Keyboard|    NULL|    Missing quantity|
|    Tom Brown|  6|  700|            Mouse|       5|Delivered, no issues|
|   Mark Davis|  7|  150|        USB Cable|       2|                    |
+-------------+---+-----+-----------------+--------+--------------------+



In [0]:
df = spark.read.format("json")\
.option("mode", "FAILFAST")\
.load("/Volumes/my_files/my_schema/sample_datasets/json_files_uploaded/sales_corrupted.json")

df.show()

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-7539114221299888>, line 5
      1 df = spark.read.format("json")\
      2 .option("mode", "FAILFAST")\
      3 .load("/Volumes/my_files/my_schema/sample_datasets/json_files_uploaded/sales_corrupted.json")
----> 5 df.show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (..

In [0]:
# Preserve corrupted records
# If you need to investigate bad records rather than discard them, use permissive mode with a corrupt-record column:

df = (
    spark.read
    .format("json")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load("/Volumes/my_files/my_schema/sample_datasets/json_files_uploaded/sales_corrupted.json")
)

df.show(truncate=False)

+-------------------------------------------------------------------------------------------------------------+-------------+----+-----+-----------------+--------+-------------------------+
|_corrupt_record                                                                                              |customer_name|id  |price|product          |quantity|remarks                  |
+-------------------------------------------------------------------------------------------------------------+-------------+----+-----+-----------------+--------+-------------------------+
|NULL                                                                                                         |John Doe     |1   |85000|Laptop, 16GB RAM |2       |Delivered on time        |
|NULL                                                                                                         |Jane Smith   |2   |55000|Smartphone, 128GB|1       |Customer said "excellent"|
|{"id":3,"customer_name":"Bob Lee","product" null,

In [0]:
# Now read a CSV file with corrupted records

df = spark.read.format("csv")\
    .load("/Volumes/my_files/my_schema/sample_datasets/csv_uploaded/sales_corrupted.csv")


df.show()


+---+----------+-----------------+----+-----+--------------------+
|_c0|       _c1|              _c2| _c3|  _c4|                 _c5|
+---+----------+-----------------+----+-----+--------------------+
|  1|  John Doe| Laptop, 16GB RAM|   2|85000|   Delivered on time|
|  2|Jane Smith|Smartphone, 128GB|   1|55000|"Customer said ""...|
|  3|   Bob Lee|             NULL|   3| 1200|                NULL|
|  4|Alice Wong|         Keyboard|NULL| 2500|                NULL|
|  5|      NULL| Monitor, 27-inch|   2| NULL|       No price info|
|  6| Tom Brown|            Mouse|  xx|  700|Delivered, no issues|
|  7|Mark Davis|        USB Cable|   2|  150|                NULL|
+---+----------+-----------------+----+-----+--------------------+



In [0]:
# Now read a CSV file with corrupted records and with a schema

schema_str = "id INT, custome_name STRING, product STRING, quantity INT, price DOUBLE, remarks STRING, _corrupted_record STRING"

# _corrupted_record STRING - This had to be added to the schema explicitly, otherwise it will be dropped by default and not created as a column
# This is because the default behavior is to drop columns that are not in the schema

df = spark.read.format("csv")\
    .schema(schema_str)\
    .load("/Volumes/my_files/my_schema/sample_datasets/csv_uploaded/sales_corrupted.csv")


df.show()

+---+------------+-----------------+--------+-------+--------------------+-----------------+
| id|custome_name|          product|quantity|  price|             remarks|_corrupted_record|
+---+------------+-----------------+--------+-------+--------------------+-----------------+
|  1|    John Doe| Laptop, 16GB RAM|       2|85000.0|   Delivered on time|             NULL|
|  2|  Jane Smith|Smartphone, 128GB|       1|55000.0|"Customer said ""...|             NULL|
|  3|     Bob Lee|             NULL|       3| 1200.0|                NULL|             NULL|
|  4|  Alice Wong|         Keyboard|    NULL| 2500.0|                NULL|             NULL|
|  5|        NULL| Monitor, 27-inch|       2|   NULL|       No price info|             NULL|
|  6|   Tom Brown|            Mouse|    NULL|  700.0|Delivered, no issues|             NULL|
|  7|  Mark Davis|        USB Cable|       2|  150.0|                NULL|             NULL|
+---+------------+-----------------+--------+-------+-----------------

In [0]:
# Now read a CSV file with corrupted records and with a schema
# Option 2

schema_str = "id INT, custome_name STRING, product STRING, quantity INT, price DOUBLE, remarks STRING, _corrupted_record STRING"

# _corrupted_record STRING - This had to be added to the schema explicitly, otherwise it will be dropped by default and not created as a column
# This is because the default behavior is to drop columns that are not in the schema

df = spark.read.format("csv")\
    .schema(schema_str)\
    .option("badrecordspath", "/Volumes/my_files/my_schema/sample_datasets/badRecordsPath")\
    .load("/Volumes/my_files/my_schema/sample_datasets/csv_uploaded/sales_corrupted.csv")


df.show()

+---+------------+-------+--------+-----+-------+-----------------+
| id|custome_name|product|quantity|price|remarks|_corrupted_record|
+---+------------+-------+--------+-----+-------+-----------------+
+---+------------+-------+--------+-----+-------+-----------------+



In [0]:
# Read the corrupted records captured by Spark
bad_df = spark.read.json("/Volumes/my_files/my_schema/sample_datasets/badRecordsPath/*/*/*")
bad_df.show(truncate=False)


+---------------------------------------------------------------------------------+------------------------------------------------------------+----------------------------------------------------------------------+
|path                                                                             |reason                                                      |record                                                                |
+---------------------------------------------------------------------------------+------------------------------------------------------------+----------------------------------------------------------------------+
|dbfs:/Volumes/my_files/my_schema/sample_datasets/csv_uploaded/sales_corrupted.csv|org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper|1,John Doe,"Laptop, 16GB RAM",2,85000,"Delivered on time"             |
|dbfs:/Volumes/my_files/my_schema/sample_datasets/csv_uploaded/sales_corrupted.csv|org.apache.spark.sql.catalyst.util.LazyBadRecordCause